# ベンダーズ分解 — 適用前・原理・適用・効果

「設計/配置を決める部分」(施設を開くか)と「その決定を所与にした運用を決める部分」(輸送)
に自然に分かれるモデルは、両方を1つのMIPに詰め込んで一括で解く（monolithic）代わりに、
**主問題(少数の連結変数)** と **サブ問題(残り、通常はLP)** に分けて交互に解ける。
サブ問題の双対から得た「感度」だけを主問題にカットとして教え、往復しながら下界と上界を
収束させる — というのがベンダーズ分解。

この notebook は次の型でこの現象と対処を追う。

1. **課題(before)** — 施設配置を1つのMIPとして monolithic に解く（結合制約の構造を確認）
2. **原理(principle)** — 主問題↔サブ問題のカット往復を実測データの収束曲線で見る
3. **適用(how)** — `mk.benders(master_build, subproblem_solve)` を最小構成で確認する
4. **効果(before/after)** — monolithic 直接解 vs ベンダーズの最終主問題(全カット搭載)を比較する

題材は **容量制約付き施設配置**(`samples/location_and_network_design/facility.py` と同じ
数理構造: 開設 $y_i \in \{0,1\}$ / 輸送 $x_{ij} \ge 0$)。ただし同梱サンプルは4施設5顧客と小さすぎて
（診断の `decomposable` 発火条件 `max_linking_groups>=4 and n_heavy_linking<=3` も満たさず、
サブ問題が実行可能でない `y` の組もある規模のため）、**同じ数理構造で規模を制御できる合成
インスタンス**をこの notebook 内で生成して検証する（CLAUDE.md方針: 確認できる問題を自分で
作って検証する）。

In [1]:
import sys, pathlib
# リポジトリルート(pyproject.toml を持つ階層)を探して import パスに載せる。
# docs/samples/ が存在するため "samples" 有無では docs で止まる。pyproject.toml を目印にする。
ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
for sub in ["samples", "samples/location_and_network_design"]:
    p = str(ROOT / sub)
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML, display
from pyscipopt import Model, quicksum
from scipy.optimize import linprog

def show(fig):  # 静的サイトに埋め込める形でグラフを表示(CDN の plotly.js を読む)
    display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False,
                             config={"displayModeBar": False})))

import minlpkit as mk
import facility as fac  # 同梱サンプル(数理構造の参照。実solveはこのnotebook内の合成データで行う)

# dataviz 参照パレット(minlpkit.live.plots と統一)
C = dict(ink="#0b0b0b", ink2="#52514e", muted="#898781", grid="#e1e0d9",
         axis="#c3c2b7", surface="#fcfcfb", s1="#2a78d6", s2="#008300", warn="#c25a00")
FONT = 'system-ui, -apple-system, "Segoe UI", sans-serif'

def base_layout(title, xtitle, ytitle, height=380):
    ax = dict(gridcolor=C["grid"], linecolor=C["axis"],
              tickfont=dict(color=C["muted"], size=11),
              title_font=dict(color=C["ink2"], size=12), zeroline=False)
    return go.Layout(
        title=dict(text=title, font=dict(color=C["ink"], size=15, family=FONT), x=0.01),
        paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
        font=dict(family=FONT, color=C["ink2"], size=12),
        xaxis=dict(ax, title=xtitle), yaxis=dict(ax, title=ytitle),
        margin=dict(l=60, r=20, t=48, b=48), height=height, hovermode="closest",
        legend=dict(orientation="h", yanchor="bottom", y=1.0, x=1.0, xanchor="right",
                    font=dict(size=11, color=C["ink2"]), bgcolor="rgba(0,0,0,0)"))
print("repo root:", ROOT)

repo root: C:\work_space\mip


## 1. 課題(before) — monolithic に施設配置を直接解く

`facility.py` と同じ構造(開設費 `fixed_i` / 容量 `cap_i` / 輸送費 `cost_ij` / 需要 `demand_j`)で、
14施設・20顧客・開設上限6のインスタンスを合成する(乱数シード固定で再現可能)。容量は
「開設上限6施設のどの組み合わせでも総容量が総需要を上回る」よう安全マージンを取って生成する
(でないとベンダーズがカットなしで初手に実行不能な `y` を選び得るため。これは打ち手の仕組みで
再度触れる `mk.benders` の制約 = 実行可能性カットを扱わない、に対応)。

In [2]:
def gen_facility(n_fac=14, n_cust=20, open_max=6, seed=0):
    rng = np.random.default_rng(seed)
    demand = rng.integers(10, 40, n_cust)
    total_demand = int(demand.sum())
    cap_center = total_demand / open_max * 1.35
    cap = rng.integers(int(cap_center * 0.85), int(cap_center * 1.15), n_fac)
    fixed = rng.integers(80, 220, n_fac)
    cost = rng.integers(1, 15, size=(n_fac, n_cust))
    facs = [f"F{i}" for i in range(n_fac)]
    custs = [f"C{j}" for j in range(n_cust)]
    return dict(facs=facs, custs=custs, fixed=dict(zip(facs, fixed)),
                cap=dict(zip(facs, cap)), demand=dict(zip(custs, demand)),
                cost={(facs[i], custs[j]): int(cost[i, j])
                      for i in range(n_fac) for j in range(n_cust)},
                open_max=open_max)


def build_monolithic(data):
    m = Model("facility_mono")
    facs, custs = data["facs"], data["custs"]
    y = {i: m.addVar(vtype="B", name=f"y_{i}") for i in facs}
    x = {(i, j): m.addVar(lb=0, name=f"x_{i}_{j}") for i in facs for j in custs}
    for j in custs:
        m.addCons(quicksum(x[i, j] for i in facs) >= data["demand"][j], name=f"demand_{j}")
    for i in facs:
        m.addCons(quicksum(x[i, j] for j in custs) <= data["cap"][i] * y[i], name=f"capacity_{i}")
    m.addCons(quicksum(y[i] for i in facs) <= data["open_max"], name="open_limit")
    m.setObjective(quicksum(data["fixed"][i] * y[i] for i in facs)
                    + quicksum(data["cost"][i, j] * x[i, j] for i in facs for j in custs), "minimize")
    m.data = dict(x=x, y=y)
    return m


data = gen_facility()
mono = build_monolithic(data)
mono.hideOutput()
mono.setParam("limits/time", 30)
mono.optimize()
mono_obj, mono_nodes, mono_time = mono.getObjVal(), mono.getNNodes(), mono.getSolvingTime()
print(f"monolithic: status={mono.getStatus()} obj={mono_obj:.1f} "
      f"nodes={mono_nodes} time={mono_time:.3f}s")

monolithic: status=optimal obj=1537.0 nodes=1 time=0.000s


この規模でも SCIP はルートノードだけで最適解を見つける(小規模なので monolithic 自体が
遅いわけではない — FINDINGS の指摘通り「小規模なら単一問題を直接解く方が速いこともある」)。
それでも構造は明確に2ブロック(開設 $y$ / 施設ごとに独立な輸送 $x_{i\cdot}$)であり、`mk.analyze` の
ブロック構造指標でも結合の様子が見える。

In [3]:
report = mk.analyze(lambda: build_monolithic(data), name="facility-synth", time_limit=15)
print(report.summary())
print("max_linking_groups:", report.metrics.get("max_linking_groups"),
      " n_heavy_linking:", report.metrics.get("n_heavy_linking"),
      "(decomposable finding は n_heavy_linking<=3 で発火。稠密な費用行列だとこの規模でも"
      "満たされないが、開設/輸送という2ブロックの分割自体は自明)")

[facility-synth] 検出症状 1件:
  [warning] 係数の絶対値レンジが桁違い / Big-M候補あり(presolve後も残存) -> スケーリング、Big-MのIndicator/SOS制約化、係数の再定式化
max_linking_groups: 21  n_heavy_linking: 14 (decomposable finding は n_heavy_linking<=3 で発火。稠密な費用行列だとこの規模でも満たされないが、開設/輸送という2ブロックの分割自体は自明)


## 2. 原理(principle) — カット往復ループの実測収束曲線

主問題は開設 $y$(整数)と輸送費の下界 $\eta$ を持つ: $\min \sum_i \mathrm{fixed}_i\, y_i + \eta$。
サブ問題は $\hat y$ を固定した輸送LP。サブ問題の容量制約の双対 $g_i$ から
**最適性カット** $\eta \ge Q(\hat y) + \sum_i g_i (y_i - \hat y_i)$ を作って主問題に足す。

```
  主問題 min c·y+η ──ŷ固定──▶ サブ問題 min 輸送費 Q(ŷ)
       ▲                              │
       └──── η ≥ Q(ŷ)+Σg·(y-ŷ) ───────┘
  下界(主問題目的) と 上界(実際のQ) が一致するまで繰り返す
```

輸送LPは施設ごとの**アグリゲート容量制約のみ**(アーク単位の上限なし)なので、
「総容量 ≥ 総需要」であれば必ず実行可能という古典的な輸送問題の可解性定理が使える。
これを主問題に直接足しておく($\sum_i \mathrm{cap}_i\, y_i \ge \text{総需要}$)ことで、`mk.benders` が扱わない
実行可能性カットなしでも安全に反復できる。

In [4]:
def make_master_build(data):
    facs = data["facs"]
    total_demand = sum(data["demand"].values())

    def master_build(cuts):
        m = Model("facility_master")
        y = {i: m.addVar(vtype="B", name=f"y_{i}") for i in facs}
        eta = m.addVar(lb=0, name="eta")
        m.addCons(quicksum(y[i] for i in facs) <= data["open_max"], name="open_limit")
        m.addCons(quicksum(data["cap"][i] * y[i] for i in facs) >= total_demand, name="agg_capacity")
        for k, cut in enumerate(cuts):
            m.addCons(eta >= cut["Q"] + quicksum(cut["grad"][i] * (y[i] - cut["yhat"][i])
                                                 for i in facs), name=f"cut_{k}")
        m.setObjective(quicksum(data["fixed"][i] * y[i] for i in facs) + eta, "minimize")
        m.data = dict(eta=eta, y=y)
        return m
    return master_build


def make_subproblem_solve(data):
    facs, custs = data["facs"], data["custs"]
    n, p = len(facs), len(custs)

    def subproblem_solve(y_hat):
        c = np.array([data["cost"][i, j] for i in facs for j in custs], dtype=float)
        A_ub = np.zeros((p + n, n * p))
        b_ub = np.zeros(p + n)
        idx = lambda a, b: a * p + b
        for jb, j in enumerate(custs):
            for ia in range(n):
                A_ub[jb, idx(ia, jb)] = -1.0
            b_ub[jb] = -data["demand"][j]
        for ia, i in enumerate(facs):
            for jb in range(p):
                A_ub[p + ia, idx(ia, jb)] = 1.0
            b_ub[p + ia] = data["cap"][i] * y_hat[i]
        res = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=(0, None), method="highs")
        Q = float(res.fun)
        duals_cap = res.ineqlin.marginals[p:p + n]
        grad = {i: float(data["cap"][i] * duals_cap[ia]) for ia, i in enumerate(facs)}
        return Q, grad

    return subproblem_solve


# master_build を呼ぶたびに直前の cuts を記録しておく(mk.benders は cuts 自体を返さないため、
# 4節で「全カット搭載の最終主問題」を再構築するのに使う)
_captured_cuts = []
def _master_build_capturing(cuts):
    _captured_cuts.clear()
    _captured_cuts.extend(cuts)
    return make_master_build(data)(cuts)

subproblem_solve = make_subproblem_solve(data)
res = mk.benders(_master_build_capturing, subproblem_solve, max_iter=200, tol=1e-6)
print(f"benders: lb={res['lb']:.1f} ub={res['ub']:.1f} cuts={res['n_cuts']} "
      f"iters={len(res['history'])}  monolithicと一致={abs(res['ub']-mono_obj) < 1e-4}")

benders: lb=1537.0 ub=1537.0 cuts=64 iters=65  monolithicと一致=True


In [5]:
its = [h["iter"] for h in res["history"]]
fig = go.Figure(layout=base_layout(
    "ベンダーズ反復の収束 — 下界(主問題)が最適性カットで上界に到達", "反復(=追加した最適性カット数)",
    "目的関数値(総費用)", height=420))
fig.add_trace(go.Scatter(x=its, y=[h["ub"] for h in res["history"]], mode="lines+markers",
    name="上界 UB(最良の実行可能解)", line=dict(color=C["warn"], width=2),
    marker=dict(size=6, color=C["warn"]),
    hovertemplate="反復%{x}<br>UB %{y:.1f}<extra></extra>"))
fig.add_trace(go.Scatter(x=its, y=[h["lb"] for h in res["history"]], mode="lines+markers",
    name="下界 LB(主問題)", line=dict(color=C["s1"], width=2),
    marker=dict(size=6, color=C["s1"]),
    hovertemplate="反復%{x}<br>LB %{y:.1f}<extra></extra>"))
fig.add_hline(y=mono_obj, line=dict(color=C["muted"], width=2, dash="dash"),
              annotation_text=f"monolithicの最適値 {mono_obj:.0f}", annotation_position="right",
              annotation_font=dict(color=C["ink2"], size=11))
show(fig)

LB(主問題、カットが増えるほど締まっていく)と UB(その時点の最良のサブ費用)が挟み込むように
monolithicの最適値へ収束する(実測値は次のセルの出力を参照)。カット1本ごとに1回の輸送LPしか
解いておらず、主問題は常に開設変数14個 + $\eta$ だけの小さなMIPのまま。

In [6]:
print(f"収束: {len(res['history'])}反復・{res['n_cuts']}カットで "
      f"LB={res['lb']:.1f} = UB={res['ub']:.1f} = monolithic最適値{mono_obj:.1f}")

収束: 65反復・64カットで LB=1537.0 = UB=1537.0 = monolithic最適値1537.0


## 3. 適用(how) — `mk.benders(master_build, subproblem_solve)`

問題固有の部分は2つのコールバックだけ: `master_build(cuts) -> Model`(`model.data` に
`eta`/`y` を入れる)と `subproblem_solve(y_hat) -> (Q, grad)`。ここでは API の契約を
最小のトイ例(2施設・2顧客)で確認する — 容量が常に足りる自明な設定で、`mk.benders` の
結果が monolithic 直接解と一致することだけを見る。

In [7]:
toy = gen_facility(n_fac=2, n_cust=2, open_max=1, seed=1)
toy_mono = build_monolithic(toy)
toy_mono.hideOutput(); toy_mono.optimize()

toy_res = mk.benders(make_master_build(toy), make_subproblem_solve(toy), max_iter=50, tol=1e-6)
print(f"toy monolithic obj={toy_mono.getObjVal():.2f}  "
      f"toy benders ub={toy_res['ub']:.2f}  一致={abs(toy_mono.getObjVal()-toy_res['ub']) < 1e-4}  "
      f"(反復{len(toy_res['history'])}・カット{toy_res['n_cuts']})")

toy monolithic obj=321.00  toy benders ub=321.00  一致=True  (反復3・カット2)

## 4. 効果(before/after) — monolithic vs 「全カット搭載の最終主問題」

`mk.compare_variants` に、(a) monolithic を直接解く build_fn と (b) 3節で収集した最終カット
一式を張った主問題(輸送変数280個を持たず、開設変数14個+$\eta$だけ)を build_fn として渡し、
ルート双対境界・最終gap・ノード数を比較する。「主問題が小さくなること」自体は自明に効くが、
カットだけで**ルートから**最適に到達できるかは別問題(カットは特定の整数点で作った支持超平面
の集まりであり、その凸包がLP緩和でどこまでタイトかは保証されない)。ここでは両方を正直に測る。

In [8]:
final_cuts = list(_captured_cuts)

def build_master_with_final_cuts():
    return make_master_build(data)(final_cuts)

df = mk.compare_variants(
    {"monolithic": lambda: build_monolithic(data),
     "benders最終主問題(全カット搭載)": build_master_with_final_cuts},
    time_limit=30)
df[["variant", "root_dual", "final_dual", "final_gap", "nodes"]]

,variant,root_dual,final_dual,final_gap,nodes
0,monolithic,1537.000000,1537.0,0.009759,1
1,benders最終主問題(全カット搭載),1347.975852,1537.0,0.000000,109


In [9]:
mono_row = df.loc[df["variant"] == "monolithic"].iloc[0]
bend_row = df.loc[df["variant"] == "benders最終主問題(全カット搭載)"].iloc[0]

fig = make_subplots(rows=1, cols=3, horizontal_spacing=0.12,
    subplot_titles=("ルート双対境界 (高いほど良い)", "最終 gap [%] (低いほど良い)",
                    "探索ノード数 (少ないほど良い)"))
labels = ["monolithic", "benders最終主問題"]
bar_colors = [C["muted"], C["s1"]]
def add_bars(col, values, fmt):
    fig.add_trace(go.Bar(x=labels, y=values, marker_color=bar_colors,
        text=[fmt(v) for v in values], textposition="outside",
        cliponaxis=False, showlegend=False), row=1, col=col)
add_bars(1, [mono_row["root_dual"], bend_row["root_dual"]], lambda v: f"{v:.0f}")
add_bars(2, [mono_row["final_gap"]*100, bend_row["final_gap"]*100], lambda v: f"{v:.1f}%")
add_bars(3, [mono_row["nodes"], bend_row["nodes"]], lambda v: f"{int(v)}")
fig.update_layout(paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
    font=dict(family=FONT, color=C["ink2"], size=12), height=360,
    margin=dict(l=40, r=20, t=64, b=40),
    title=dict(text="ベンダーズ分解の before / after(monolithic vs 全カット搭載の最終主問題)",
               x=0.01, font=dict(color=C["ink"], size=15)))
fig.update_yaxes(gridcolor=C["grid"], linecolor=C["axis"], zeroline=False)
fig.update_xaxes(linecolor=C["axis"])
show(fig)

In [10]:
print(f"ルート双対境界 : monolithic {mono_row['root_dual']:.1f}  vs  "
      f"benders最終主問題 {bend_row['root_dual']:.1f}")
print(f"最終gap        : monolithic {mono_row['final_gap']:.1%}  vs  "
      f"benders最終主問題 {bend_row['final_gap']:.1%}")
print(f"ノード数       : monolithic {int(mono_row['nodes'])}  vs  "
      f"benders最終主問題 {int(bend_row['nodes'])}")
print("この規模では monolithic がルート1ノードで最適に達し、"
      "カット搭載の主問題(変数は14+eta、輸送変数280個を持たない)はカットの凸包が"
      "ルートLPではタイトでないぶん、逆に分枝(109ノード)を要している。")

ルート双対境界 : monolithic 1537.0  vs  benders最終主問題 1348.0
最終gap        : monolithic 1.0%  vs  benders最終主問題 0.0%
ノード数       : monolithic 1  vs  benders最終主問題 109
この規模では monolithic がルート1ノードで最適に達し、カット搭載の主問題(変数は14+eta、輸送変数280個を持たない)はカットの凸包がルートLPではタイトでないぶん、逆に分枝(109ノード)を要している。


## まとめ

- **主問題から輸送変数(14施設×20顧客=280個の連続変数)を完全に追い出しても、
  カットの往復だけで monolithic と同じ最適値 1537 に厳密収束する**(2節の収束曲線)。
  これがベンダーズの核心的な効果であり、この instance でも成立している。
- 一方で「カット搭載の主問題を単体で解いたときのノード数」は、必ずしも monolithic を
  下回らない(4節)。カットは特定の整数 `ŷ` で作った支持超平面の集まりに過ぎず、
  その凸包が LP 緩和(ルート)でどこまでタイトかは構造依存。今回のインスタンスでは
  monolithic 自身がルート1ノードで解けてしまうほど「簡単」だったため、
  カット主問題の分枝コストが相対的に見えている。

### なぜ SCIP が自動でやらないのか

SCIP は与えられたMIPをそのまま(Big-M・カット・分枝で)解く。「連結変数とサブ問題に
問題を分割し、サブの双対を主問題にカットとしてフィードバックする」という**分解の切り口
そのもの**はモデラーが決める設計判断であり、build 済みの1つのMIPからは自動抽出できない。
だから診断がブロック構造(結合制約の少なさ)を指摘し、モデラーが `master_build`/
`subproblem_solve` を書く、という分業になる。

### 効かない条件・注意

- 実装した `mk.benders` は**実行可能性カットを扱わない**。この notebook では「アグリゲート
  容量 ≥ 総需要」を主問題に直接張ることでこれを回避した(輸送LPがアーク単位の上限を
  持たない特殊構造だからこそ可能な簡略化)。より一般の構造(アーク容量あり等)では
  サブ問題が実行可能でない `ŷ` が起こりうるため拡張が要る。
- 小規模なら monolithic を直接解く方が速いこともある(本 notebook のインスタンスでも
  monolithic はルートノードで解けている)。分解の価値は「主問題を小さく保てる」
  「サブ問題を独立に(並列に)解ける」という**スケールする性質**にある。

関連: [手法ガイド 5. ベンダーズ分解](../../playbook/05-benders.md) /
API [`mk.benders`](../../api/frameworks.md) / worked example: `experiments/run_benders.py`